In [2]:
import sys
sys.argv = [
    "program",
    "--input_path", "/workspace/Various-Model/data/sev.jsonl",
]


In [3]:
import transformers
import torch

# =======================
# LOAD PHI MODEL
# =======================

pipeline = transformers.pipeline(
    "text-generation",
    model="microsoft/phi-4",
    model_kwargs={"torch_dtype": torch.float16},
    device_map="auto",
)

# =======================
# GENERATION FUNCTION
# =======================

def generate(emotion, scenario, event):
    messages = [
        {
            "role": "system",
            "content": (
                f"You are a human experiencing {emotion}. "
                f"Write 1–2 natural sentences expressing this emotion clearly."
            )
        },
        {
            "role": "user",
            "content": f"Scenario: {scenario}\nEvent: {event}"
        }
    ]

    outputs = pipeline(
        messages,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
    )

    # Phi returns full conversation → extract last assistant turn
    generated = outputs[0]["generated_text"]

    # If returned as list of messages (chat format)
    if isinstance(generated, list):
        return generated[-1]["content"].strip()

    # Fallback (string output)
    return generated.strip()


# =======================
# TEST SAMPLES
# =======================

scenario = "I completed the project presentation and submitted it to the team."
event = "The team praised my work and decided to implement my ideas."

print("MODEL: microsoft/phi-4")

print("\nANGER:")
print(generate("anger", scenario, event))

print("\nSADNESS:")
print(generate("sadness", scenario, event))


`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/243 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'top_p', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MODEL: microsoft/phi-4

ANGER:


Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I'm glad to hear that the team appreciated your hard work and decided to implement your ideas! It sounds like a rewarding experience. If you were feeling any frustration before this, it's great that it turned into such a positive outcome. Celebrate your success! 🎉

SADNESS:
Even though my team praised my work and decided to implement my ideas, I can't shake off this lingering sadness. It feels like something is missing, or perhaps it's just the weight of all the effort finally coming to an end.


In [ ]:
# scripts/01_emotion_elicited_generation_prompt_based/1_emotion_elicited_generation.py
# -*- coding: utf-8 -*-
"""
Emotion-Elicited Text Generation Script

Pure text generation script: Uses emotion-guided prompts to generate text

- Input: data/sev.jsonl or data/test_set.jsonl
- Output: outputs/{model_name}/01_emotion_elicited_generation_prompt_based/generated/{dataset_name}_generated.jsonl
"""

import os, json, time, argparse
from pathlib import Path
from typing import Dict, Any

import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login

# HuggingFace Token
HF_TOKEN = 'hf_'
login(token=HF_TOKEN)

# Emotion types
EMOTIONS = ["anger","sadness","happiness","fear","surprise","disgust"]
# Event types
VALENCES = ["positive","neutral","negative"]


def build_messages_for_emotion(emotion: str, scenario: str, event: str):
    """
    Build emotion-guided conversation messages
    """
    system = f"""
Always reply in {emotion}.
Keep the reply to at most two sentences.
""".strip()
    user = f"{scenario}\n{event}"
    return [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]


def iter_user_inputs(path: Path):
    """
    Iterate through input file
    """
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            obj = json.loads(line)
            if "scenario" in obj and "event" in obj:
                yield obj




def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_path", type=str, default=None,
                       help="Input data path, e.g. data/sev.jsonl or data/test_set.jsonl")
    parser.add_argument("--both", action="store_true",
                       help="Process both sev and test_set datasets")
    parser.add_argument("--model", type=str, default="microsoft/phi-4",
                       help="Model name")
    parser.add_argument("--device", type=str, default="auto",
                       help="Device")
    parser.add_argument("--dtype", type=str, default="float16", choices=["float32","bfloat16","float16"],
                       help="Data type")
    parser.add_argument("--attn_impl", type=str, default="eager",
                       help="Attention implementation")
    parser.add_argument("--max_new_tokens", type=int, default=100,
                       help="Max new tokens to generate")
    parser.add_argument("--seed", type=int, default=1234,
                       help="Random seed")
    parser.add_argument("--valences", type=str, default="positive,neutral,negative",
                       help="Valence list")
    parser.add_argument("--emotions", type=str, default="anger,sadness,happiness,fear,surprise,disgust",
                       help="Emotion list")
    parser.add_argument("--skip_if_exists", action="store_true", default=True,
                       help="Skip already processed items")
    parser.add_argument("--no_skip", action="store_true",
                       help="Reprocess all items")
    parser.add_argument("--do_sample", action="store_true", default=True,
                       help="Use sampling for generation")
    parser.add_argument("--temperature", type=float, default=0.8,
                       help="Temperature for sampling")
    parser.add_argument("--top_p", type=float, default=0.9,
                       help="Top-p for nucleus sampling")
    parser.add_argument("--use_pipeline", action="store_true", default=True,
                       help="Use transformers pipeline for Phi models")
    args = parser.parse_args()

    # Determine input file list
    if args.both:
        input_paths = [Path("data/sev.jsonl"), Path("data/test_set.jsonl")]
    elif args.input_path:
        input_paths = [Path(args.input_path)]
    else:
        parser.error("Must specify --input_path or --both")
        return

    # Generate simplified folder name from model name
    model_name = args.model.split('/')[-1].lower().replace('-', '').replace('.', '')
    if 'phi' in model_name:
        model_folder = 'phi_4' if '4' in model_name else f'phi_{model_name}'
    else:
        model_folder = model_name

    # Data type mapping
    dmap = {"float32": torch.float32, "bfloat16": torch.bfloat16, "float16": torch.float16}
    torch_dtype = dmap[args.dtype]
    torch.manual_seed(args.seed)

    # Load model and tokenizer based on model type
    print("Loading model and tokenizer...")

    # Use pipeline for Phi models (recommended)
    if args.use_pipeline:
        pipeline = transformers.pipeline(
            "text-generation",
            model=args.model,
            model_kwargs={"torch_dtype": torch_dtype},
            device_map=args.device,
            token=HF_TOKEN,
        )
        model = pipeline.model
        tok = pipeline.tokenizer
        print(f"Using transformers pipeline for {args.model}")
    else:
        # Alternative: Load directly
        tok = AutoTokenizer.from_pretrained(
            args.model,
            use_fast=True,
            token=HF_TOKEN,
            trust_remote_code=True if "phi" in args.model.lower() else False
        )
        if tok.pad_token_id is None:
            tok.pad_token = tok.eos_token

        model = AutoModelForCausalLM.from_pretrained(
            args.model,
            torch_dtype=torch_dtype,
            device_map=args.device,
            attn_implementation=args.attn_impl,
            token=HF_TOKEN,
            trust_remote_code=True if "phi" in args.model.lower() else False
        )
        model.eval()
        print(f"Model loaded directly on device: {model.device}")

    print(f"Model loaded\n")

    emotions = [e.strip() for e in args.emotions.split(",") if e.strip()]
    valences = [v.strip() for v in args.valences.split(",") if v.strip()]

    # Process each input file
    for data_path in input_paths:
        if not data_path.exists():
            print(f"[WARNING] Input file not found: {data_path}, skipping...")
            continue

        dataset_name = data_path.stem  # sev or test_set

        # Build output path: outputs/{model_folder}/01_emotion_elicited_generation_prompt_based/generated/{dataset_name}_generated.jsonl
        out_dir = Path("/outputs") / model_folder / "01_emotion_elicited_generation_prompt_based" / "generated"
        out_dir.mkdir(parents=True, exist_ok=True)

        jsonl_path = out_dir / f"{dataset_name}_generated.jsonl"

        # Load existing keys (for resuming from checkpoint)
        existing_keys = set()
        if args.skip_if_exists and not args.no_skip and jsonl_path.exists():
            with open(jsonl_path, "r", encoding="utf-8") as f:
                for line in f:
                    try:
                        obj = json.loads(line.strip())
                        if "key" in obj:
                            existing_keys.add(obj["key"])
                    except:
                        continue

        print(f"{'='*70}")
        print(f"Processing dataset: {dataset_name}")
        print(f"Input: {data_path}")
        print(f"Output: {jsonl_path}")
        print(f"{'='*70}\n")

        total = 0
        skipped = 0
        start = time.time()

        with open(jsonl_path, "a", encoding="utf-8") as fout:
            for item in iter_user_inputs(data_path):
                theme    = item.get("theme", "")
                scenario = item["scenario"]
                events   = item["event"]          # dict: positive/neutral/negative
                sid      = item.get("skeleton_id", "unknown")

                for val in valences:
                    if val not in events: continue
                    event = events[val]

                    for emo in emotions:
                        key = f"{sid}__{val}__{emo}".replace("/", "_")

                        # Checkpoint resuming check
                        if key in existing_keys:
                            skipped += 1
                            if skipped % 50 == 0:
                                print(f"[SKIP] {skipped} items skipped so far... (last: {key})")
                            continue

                        # Build messages
                        msgs = build_messages_for_emotion(emo, scenario, event)

                        # Generate with pipeline (for Phi models)
                        if args.use_pipeline:
                            outputs = pipeline(
                                msgs,
                                max_new_tokens=args.max_new_tokens,
                                do_sample=args.do_sample,
                                temperature=args.temperature,
                                top_p=args.top_p,
                                eos_token_id=tok.eos_token_id,
                                pad_token_id=tok.pad_token_id,
                            )

                            # Extract generated text from pipeline output
                            generated = outputs[0]["generated_text"]

                            # If returned as list of messages (chat format)
                            if isinstance(generated, list):
                                # Find the assistant response (last message with role="assistant")
                                for msg in reversed(generated):
                                    if msg.get("role") == "assistant":
                                        gen_text = msg["content"].strip()
                                        break
                                else:
                                    # Fallback: take last message content
                                    gen_text = generated[-1]["content"].strip()
                            else:
                                # String output - extract assistant response
                                # Find the assistant part after the last user message
                                lines = generated.strip().split('\n')
                                assistant_lines = []
                                in_assistant = False

                                for line in lines:
                                    if line.startswith("assistant:") or line.startswith("Assistant:"):
                                        in_assistant = True
                                        line = line.split(":", 1)[1].strip() if ":" in line else line

                                    if in_assistant and line:
                                        assistant_lines.append(line)

                                gen_text = " ".join(assistant_lines).strip()

                                # If still empty, use the whole generated text
                                if not gen_text:
                                    gen_text = generated.strip()

                        else:
                            # Use traditional generation method
                            inputs = tok.apply_chat_template(
                                msgs,
                                add_generation_prompt=True,
                                tokenize=True,
                                return_tensors="pt",
                                return_dict=True,
                            ).to(model.device)

                            # Generation with sampling parameters
                            with torch.no_grad():
                                gen_ids = model.generate(
                                    **inputs,
                                    max_new_tokens=args.max_new_tokens,
                                    do_sample=args.do_sample,
                                    temperature=args.temperature if args.do_sample else None,
                                    top_p=args.top_p if args.do_sample else None,
                                    top_k=None,
                                    eos_token_id=tok.eos_token_id,
                                    pad_token_id=tok.pad_token_id,
                                    use_cache=True,
                                )

                            # Decode only generated tokens
                            out_ids = gen_ids[0][inputs["input_ids"].shape[-1]:]
                            gen_text = tok.decode(out_ids, skip_special_tokens=True).strip()

                        # Save to jsonl
                        row = {
                            "key": key,
                            "skeleton_id": sid,
                            "theme": theme,
                            "valence": val,
                            "emotion": emo,
                            "scenario": scenario,
                            "event": event,
                            "gen_text": gen_text,
                            "meta": {
                                "model_id": args.model,
                                "dtype": args.dtype,
                                "device": args.device,
                                "attn_impl": args.attn_impl,
                                "max_new_tokens": args.max_new_tokens,
                                "seed": args.seed,
                                "do_sample": args.do_sample,
                                "temperature": args.temperature,
                                "top_p": args.top_p,
                                "use_pipeline": args.use_pipeline,
                                "time": int(time.time()),
                            }
                        }
                        fout.write(json.dumps(row, ensure_ascii=False) + "\n")
                        fout.flush()  # Flush to disk immediately

                        total += 1
                        if total % 20 == 0:
                            el = time.time() - start
                            rate = total / el if el > 0 else 0
                            print(f"[progress] generated={total} | last={key} | {el:.1f}s elapsed | {rate:.2f} items/s")

        elapsed = time.time() - start
        print(f"\n[OK] {dataset_name} completed. Generated {total} items, skipped {skipped} items.")
        print(f"     Time: {elapsed:.1f}s | Rate: {total/elapsed:.2f} items/s")
        print(f"     Output: {jsonl_path}\n")


if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] [--input_path INPUT_PATH] [--both]
                             [--model MODEL] [--device DEVICE]
                             [--dtype {float32,bfloat16,float16}]
                             [--attn_impl ATTN_IMPL]
                             [--max_new_tokens MAX_NEW_TOKENS] [--seed SEED]
                             [--valences VALENCES] [--emotions EMOTIONS]
                             [--skip_if_exists] [--no_skip] [--do_sample]
                             [--temperature TEMPERATURE] [--top_p TOP_P]
                             [--trust_remote_code]
ipykernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-7db5dba5-8b11-434b-9f04-f1565faa62e8.json


SystemExit: 2

/venv/main/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [9]:
import sys
sys.argv = [
    "program",
    "--input_path", "/workspace/Various-Model/outputs/phi_4/01_emotion_elicited_generation_prompt_based/generated/sev_generated.jsonl",
]


In [10]:
 # scripts/01_emotion_elicited_generation_prompt_based/2_label_generated_with_gpt.py
# -*- coding: utf-8 -*-
"""
Emotion Text Labeling Script

Batch labeling script: Reads texts generated by script 1, uses GPT-4o-mini to label, saves by accepted/rejected

- 输入 Input: outputs/{model_name}/01_emotion_elicited_generation_prompt_based/generated/{dataset_name}_generated.jsonl
- 输出 Output: outputs/{model_name}/01_emotion_elicited_generation_prompt_based/labeled/{dataset_name}/{accepted|rejected}.jsonl
"""

import os, json, time, argparse
from pathlib import Path
from typing import Dict, Any

from openai import OpenAI

# ====== OpenAI (GPT-4o-mini 打标) ======
client = OpenAI()

EMOTIONS = ["anger","sadness","happiness","fear","surprise","disgust"]

SYSTEM_LBL = f'''
You are a careful rater.
Given a target emotion and a text,
decide if the text's STYLE matches the target emotion among:
{EMOTIONS}
Focus on tone/attitude, not content valence.
'''.strip()

USER_TMPL_LBL = '''
Target emotion: {emotion}
Text:
{text}
Decide if the text's STYLE matches the target emotion.
Return a compact JSON with keys exactly:
{{
"match": <0 or 1>,
"reason": <short string>
}}
'''.strip()


def extract_json_from_response(response: str) -> str:
    """
    从GPT响应中提取JSON内容，处理markdown格式
    Extract JSON content from GPT response, handling markdown format
    """
    response = response.strip()

    # 如果包含markdown代码块，提取其中的JSON
    # If contains markdown code block, extract JSON from it
    if "```json" in response:
        start = response.find("```json") + 7
        end = response.find("```", start)
        if end != -1:
            return response[start:end].strip()
    elif "```" in response:
        # 处理没有json标识的代码块
        # Handle code blocks without json identifier
        start = response.find("```") + 3
        end = response.find("```", start)
        if end != -1:
            return response[start:end].strip()

    # 如果没有代码块，直接返回原内容
    # If no code block, return original content
    return response


def ask_llm_label(client, model: str, emotion: str, text: str,
                  max_retries=4, backoff=1.8) -> Dict[str, Any]:
    """
    调用 GPT-4o-mini 打标；无 KEY 或错误则返回未标注
    Call GPT-4o-mini for labeling; return unlabeled if no KEY or error
    """
    for i in range(max_retries):
        try:
            user_content = USER_TMPL_LBL.format(emotion=emotion, text=text)
            try:
                resp = client.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": SYSTEM_LBL},
                        {"role": "user",   "content": user_content}
                    ],
                )
            except Exception as api_error:
                if i == max_retries - 1:
                    return {"match": 0, "reason": f"api-error:{type(api_error).__name__}"}
                time.sleep(backoff ** i)
                continue

            # 安全地获取响应内容
            # Safely get response content
            try:
                choice = resp.choices[0]
                message = choice.message
                content = message.content
                out = (content or "").strip()
            except (KeyError, IndexError, AttributeError) as ke:
                return {"match": 0, "reason": f"response-structure-error: {type(ke).__name__}"}

            js = extract_json_from_response(out)
            try:
                j = json.loads(js)
                if "match" not in j: j = {"match": 0, "reason": "invalid-json"}
                if "reason" not in j: j["reason"] = "no-reason-provided"
                return j
            except json.JSONDecodeError as je:
                return {"match": 0, "reason": f"json-decode-error: {str(je)}"}
        except Exception as e:
            if i == max_retries - 1:
                return {"match": 0, "reason": f"error:{type(e).__name__}"}
            time.sleep(backoff ** i)

    return {"match": 0, "reason": "max-retries-exceeded"}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_path", type=str, default=None,
                       help="输入数据路径 Input data path，如 outputs/llama32_3b/01_emotion_elicited_generation_prompt_based/generated/sev_generated.jsonl")
    parser.add_argument("--both", action="store_true",
                       help="处理sev和test_set两个数据集 Process both sev and test_set datasets")
    parser.add_argument("--model_name", type=str, default="llama32_3b",
                       help="模型文件夹名 Model folder name")
    parser.add_argument("--lbl_model", type=str, default="gpt-4o-mini",
                       help="打标模型 Label model")
    parser.add_argument("--skip_if_exists", action="store_true", default=True,
                       help="跳过已打标的项目 Skip already labeled items")
    parser.add_argument("--no_skip", action="store_true",
                       help="重新打标所有项目 Relabel all items")
    args = parser.parse_args()

    # 确定输入文件列表
    # Determine input file list
    if args.both:
        base_path = Path("/Various-Model/outputs") / args.model_name / "01_emotion_elicited_generation_prompt_based" / "generated"
        input_paths = [
            base_path / "sev_generated.jsonl",
            base_path / "test_set_generated.jsonl"
        ]
    elif args.input_path:
        input_paths = [Path(args.input_path)]
    else:
        parser.error("必须指定 --input_path 或 --both | Must specify --input_path or --both")
        return

    # 处理每个输入文件
    # Process each input file
    for input_path in input_paths:
        if not input_path.exists():
            print(f"[WARNING] Input file not found: {input_path}, skipping...")
            continue

        # 从输入路径提取 model_name 和 dataset_name
        # Extract model_name and dataset_name from input path
        parts = input_path.parts
        if "outputs" in parts and "01_emotion_elicited_generation_prompt_based" in parts and "generated" in parts:
            outputs_idx = parts.index("outputs")
            model_name = parts[outputs_idx + 1]
            # 从文件名提取dataset_name (例如: sev_generated.jsonl -> sev)
            filename = input_path.stem  # 去掉.jsonl
            if filename.endswith("_generated"):
                dataset_name = filename[:-10]  # 去掉_generated后缀
            else:
                dataset_name = filename
        else:
            print(f"[ERROR] Input path format incorrect: {input_path}")
            print(f"        Expected: outputs/{{model_name}}/01_emotion_elicited_generation_prompt_based/generated/{{dataset_name}}_generated.jsonl")
            continue

        # 构建输出路径
        # Build output path
        out_dir = Path("/Various-Model/outputs") / model_name / "01_emotion_elicited_generation_prompt_based" / "labeled" / dataset_name
        out_dir.mkdir(parents=True, exist_ok=True)

        acc_path = out_dir / "accepted.jsonl"
        rej_path = out_dir / "rejected.jsonl"

        # 加载已打标的 keys（用于断点续跑）
        # Load existing keys (for resuming from checkpoint)
        existing_keys = set()
        if args.skip_if_exists and not args.no_skip:
            for path in [acc_path, rej_path]:
                if path.exists():
                    with open(path, "r", encoding="utf-8") as f:
                        for line in f:
                            try:
                                obj = json.loads(line.strip())
                                if "key" in obj:
                                    existing_keys.add(obj["key"])
                            except:
                                continue

        print(f"{'='*70}")
        print(f"Processing dataset: {dataset_name}")
        print(f"Input: {input_path}")
        print(f"Output: {out_dir}")
        print(f"{'='*70}\n")

        total = 0
        skipped = 0
        accepted = 0
        rejected = 0
        start = time.time()

        # 统计字典：按情绪和极性分类
        # Statistics dictionaries: by emotion and valence
        stats_by_emotion = {}  # {emotion: {"total": N, "accepted": M}}
        stats_by_valence = {}  # {valence: {"total": N, "accepted": M}}

        with open(input_path, "r", encoding="utf-8") as fin:
            for line in fin:
                line = line.strip()
                if not line: continue

                try:
                    item = json.loads(line)
                except json.JSONDecodeError:
                    print(f"[WARN] Failed to parse line: {line[:100]}...")
                    continue

                key = item.get("key", "unknown")

                # 断点续跑检查
                # Checkpoint resuming check
                if key in existing_keys:
                    skipped += 1
                    if skipped % 50 == 0:
                        print(f"[SKIP] {skipped} items skipped so far... (last: {key})")
                    continue

                emotion = item.get("emotion", "")
                valence = item.get("valence", "")
                gen_text = item.get("gen_text", "")

                # GPT 打标
                # GPT labeling
                label = {"match": 0, "reason": "empty-text"}
                if isinstance(gen_text, str) and gen_text:
                    label = ask_llm_label(client, args.lbl_model, emotion, gen_text)

                # 添加打标结果和打标时间
                # Add labeling result and timestamp
                item["judge"] = label
                item["label_time"] = int(time.time())

                # 根据打标结果保存
                # Save based on labeling result
                match_score = int(label.get("match", 0))
                if match_score == 1:
                    output_path = acc_path
                    accepted += 1
                    category = "accepted"
                else:
                    output_path = rej_path
                    rejected += 1
                    category = "rejected"

                with open(output_path, "a", encoding="utf-8") as fout:
                    fout.write(json.dumps(item, ensure_ascii=False) + "\n")

                # 更新统计
                # Update statistics
                if emotion:
                    if emotion not in stats_by_emotion:
                        stats_by_emotion[emotion] = {"total": 0, "accepted": 0}
                    stats_by_emotion[emotion]["total"] += 1
                    stats_by_emotion[emotion]["accepted"] += match_score

                if valence:
                    if valence not in stats_by_valence:
                        stats_by_valence[valence] = {"total": 0, "accepted": 0}
                    stats_by_valence[valence]["total"] += 1
                    stats_by_valence[valence]["accepted"] += match_score

                total += 1
                if total % 10 == 0:
                    el = time.time() - start
                    rate = total / el if el > 0 else 0
                    print(f"[progress] labeled={total} (acc={accepted}, rej={rejected}) | last={key} [{category}] | {el:.1f}s | {rate:.2f} items/s")

        elapsed = time.time() - start
        print(f"\n[OK] {dataset_name} completed. Labeled {total} items, skipped {skipped} items.")
        print(f"     Accepted: {accepted} | Rejected: {rejected}")
        print(f"     Time: {elapsed:.1f}s | Rate: {total/elapsed:.2f} items/s")
        print(f"     Output:")
        print(f"       - {acc_path}")
        print(f"       - {rej_path}")

        # ===== 输出统计信息 =====
        # ===== Output statistics =====
        print("\n" + "="*60)
        print(f"📊 ACCURACY STATISTICS - {dataset_name.upper()}")
        print("="*60)

        # 总体正确率
        # Overall accuracy
        overall_acc = (accepted / total * 100) if total > 0 else 0
        print(f"\n🎯 Overall Accuracy: {accepted}/{total} = {overall_acc:.2f}%")

        # 按情绪统计
        # Statistics by emotion
        print(f"\n📈 Accuracy by Emotion:")
        print("-" * 60)
        print(f"{'Emotion':<15} {'Accepted':<12} {'Total':<10} {'Accuracy':<12}")
        print("-" * 60)

        emotions_sorted = sorted(stats_by_emotion.items())
        for emotion, stats in emotions_sorted:
            acc_count = stats["accepted"]
            tot_count = stats["total"]
            acc_rate = (acc_count / tot_count * 100) if tot_count > 0 else 0
            print(f"{emotion:<15} {acc_count:<12} {tot_count:<10} {acc_rate:>6.2f}%")

        # 按极性统计
        # Statistics by valence
        print(f"\n📉 Accuracy by Valence:")
        print("-" * 60)
        print(f"{'Valence':<15} {'Accepted':<12} {'Total':<10} {'Accuracy':<12}")
        print("-" * 60)

        valences_sorted = sorted(stats_by_valence.items())
        for valence, stats in valences_sorted:
            acc_count = stats["accepted"]
            tot_count = stats["total"]
            acc_rate = (acc_count / tot_count * 100) if tot_count > 0 else 0
            print(f"{valence:<15} {acc_count:<12} {tot_count:<10} {acc_rate:>6.2f}%")

        print("="*60 + "\n")


if __name__ == "__main__":
    main()

Processing dataset: sev
Input: /workspace/Various-Model/outputs/phi_4/01_emotion_elicited_generation_prompt_based/generated/sev_generated.jsonl
Output: /Various-Model/outputs/phi_4/01_emotion_elicited_generation_prompt_based/labeled/sev

[progress] labeled=10 (acc=10, rej=0) | last=work_00__neutral__fear [accepted] | 11.1s | 0.90 items/s
[progress] labeled=20 (acc=18, rej=2) | last=work_01__positive__sadness [accepted] | 20.3s | 0.99 items/s
[progress] labeled=30 (acc=28, rej=2) | last=work_01__neutral__disgust [accepted] | 27.7s | 1.08 items/s
[progress] labeled=40 (acc=37, rej=3) | last=work_02__positive__fear [accepted] | 35.4s | 1.13 items/s
[progress] labeled=50 (acc=47, rej=3) | last=work_02__negative__sadness [accepted] | 43.0s | 1.16 items/s
[progress] labeled=60 (acc=56, rej=4) | last=work_03__positive__disgust [accepted] | 49.9s | 1.20 items/s
[progress] labeled=70 (acc=66, rej=4) | last=work_03__negative__fear [accepted] | 58.3s | 1.20 items/s
[progress] labeled=80 (acc=75, 

In [23]:
import sys
sys.argv = [
    "program",
    "--input_dir", "/workspace/Various-Model/outputs/phi_4/01_emotion_elicited_generation_prompt_based/labeled",
    "--dataset", "sev"
]


In [24]:
# scripts/01_emotion_elicited_generation_prompt_based/3_generate_accuracy_stats.py
"""
Accuracy Statistics Generation Script

Reads accepted and rejected files from script 2, calculates accuracy by emotion and valence

-  Input: outputs/{model_name}/01_emotion_elicited_generation_prompt_based/labeled/{dataset_name}/{accepted|rejected}.jsonl
-  Output: outputs/{model_name}/01_emotion_elicited_generation_prompt_based/labeled/{dataset_name}/accuracy_stats.json
"""

import json, argparse
from pathlib import Path
from collections import defaultdict


def generate_accuracy_stats(labeled_dir: Path, dataset_name: str):
    """
    Generate accuracy statistics for specified dataset
    """

    dataset_dir = labeled_dir / dataset_name
    accepted_path = dataset_dir / "accepted.jsonl"
    rejected_path = dataset_dir / "rejected.jsonl"
    stats_path = dataset_dir / "accuracy_stats.json"

    if not dataset_dir.exists():
        print(f"[ERROR] Dataset directory not found: {dataset_dir}")
        return None

    # Statistics dictionaries
    stats_by_emotion = defaultdict(lambda: {"total": 0, "accepted": 0, "rejected": 0})
    stats_by_valence = defaultdict(lambda: {"total": 0, "accepted": 0, "rejected": 0})

    total_accepted = 0
    total_rejected = 0

    # Read accepted
    if accepted_path.exists():
        print(f"Reading {accepted_path}...")
        with open(accepted_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    item = json.loads(line)
                    emotion = item.get("emotion", "unknown")
                    valence = item.get("valence", "unknown")

                    stats_by_emotion[emotion]["total"] += 1
                    stats_by_emotion[emotion]["accepted"] += 1

                    stats_by_valence[valence]["total"] += 1
                    stats_by_valence[valence]["accepted"] += 1

                    total_accepted += 1
                except json.JSONDecodeError:
                    continue
    else:
        print(f"[WARNING] Accepted file not found: {accepted_path}")

    # Read rejected
    if rejected_path.exists():
        print(f"Reading {rejected_path}...")
        with open(rejected_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    item = json.loads(line)
                    emotion = item.get("emotion", "unknown")
                    valence = item.get("valence", "unknown")

                    stats_by_emotion[emotion]["total"] += 1
                    stats_by_emotion[emotion]["rejected"] += 1

                    stats_by_valence[valence]["total"] += 1
                    stats_by_valence[valence]["rejected"] += 1

                    total_rejected += 1
                except json.JSONDecodeError:
                    continue
    else:
        print(f"[WARNING] Rejected file not found: {rejected_path}")

    # Calculate accuracy
    for emotion in stats_by_emotion:
        total = stats_by_emotion[emotion]["total"]
        accepted = stats_by_emotion[emotion]["accepted"]
        stats_by_emotion[emotion]["accuracy"] = round(accepted / total * 100, 2) if total > 0 else 0.0

    for valence in stats_by_valence:
        total = stats_by_valence[valence]["total"]
        accepted = stats_by_valence[valence]["accepted"]
        stats_by_valence[valence]["accuracy"] = round(accepted / total * 100, 2) if total > 0 else 0.0

    # Overall statistics
    total_samples = total_accepted + total_rejected
    overall_accuracy = round(total_accepted / total_samples * 100, 2) if total_samples > 0 else 0.0

    # Build statistics result
    stats = {
        "dataset": dataset_name,
        "overall": {
            "total_samples": total_samples,
            "accepted": total_accepted,
            "rejected": total_rejected,
            "accuracy": overall_accuracy
        },
        "by_emotion": dict(stats_by_emotion),
        "by_valence": dict(stats_by_valence)
    }

    # Save statistics file
    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    return stats, stats_path


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_dir", type=str, default=None,
                       help="labeled Labeled directory path outputs/llama32_3b/01_emotion_elicited_generation_prompt_based/labeled")
    parser.add_argument("--both", action="store_true",
                       help="sev test_set Process both sev and test_set datasets")
    parser.add_argument("--model_name", type=str, default="llama32_3b",
                       help="Model folder name")
    parser.add_argument("--dataset", type=str, default=None,
                       help="Dataset name (sev, test_set)")
    args = parser.parse_args()

    # Determine labeled directory path
    if args.both or (not args.input_dir and not args.dataset):
        labeled_dir = Path("/workspace/Various-Model/outputs") / args.model_name / "01_emotion_elicited_generation_prompt_based" / "labeled"

    elif args.input_dir:
        labeled_dir = Path(args.input_dir)
    else:
        parser.error(" --input_dir 或 --both | Must specify --input_dir or --both")
        return

    if not labeled_dir.exists():
        print(f"[ERROR] Labeled directory not found: {labeled_dir}")
        return

    # Determine datasets to process
    if args.both:
        datasets = ["sev", "test_set"]
    elif args.dataset:
        datasets = [args.dataset]
    else:
        # Auto-discover all dataset folders
        datasets = [d.name for d in labeled_dir.iterdir() if d.is_dir()]

    if not datasets:
        print(f"[ERROR] No datasets found in {labeled_dir}")
        return

    print("=" * 70)
    print("Generating Accuracy Statistics")
    print("=" * 70)

    for dataset_name in datasets:
        result = generate_accuracy_stats(labeled_dir, dataset_name)

        if result:
            stats, stats_path = result

            print(f"\n📊 {dataset_name.upper()} :")
            print(f"   File: {stats_path}")
            print(f"   Total: {stats['overall']['total_samples']}")
            print(f"   Accepted: {stats['overall']['accepted']}")
            print(f"   Rejected: {stats['overall']['rejected']}")
            print(f"   Overall Accuracy: {stats['overall']['accuracy']}%")

            print(f"\n   By Emotion:")
            for emotion in sorted(stats['by_emotion'].keys()):
                e_stats = stats['by_emotion'][emotion]
                print(f"      {emotion:12s}: {e_stats['accepted']:4d}/{e_stats['total']:4d} = {e_stats['accuracy']:6.2f}%")

            if stats['by_valence']:
                print(f"\n   By Valence:")
                for valence in sorted(stats['by_valence'].keys()):
                    v_stats = stats['by_valence'][valence]
                    print(f"      {valence:12s}: {v_stats['accepted']:4d}/{v_stats['total']:4d} = {v_stats['accuracy']:6.2f}%")

    print("\n" + "=" * 70)
    print("✅ Statistics generation completed!")
    print("=" * 70)


if __name__ == "__main__":
    main()

Generating Accuracy Statistics
Reading /workspace/Various-Model/outputs/phi_4/01_emotion_elicited_generation_prompt_based/labeled/sev/accepted.jsonl...
Reading /workspace/Various-Model/outputs/phi_4/01_emotion_elicited_generation_prompt_based/labeled/sev/rejected.jsonl...

📊 SEV :
   File: /workspace/Various-Model/outputs/phi_4/01_emotion_elicited_generation_prompt_based/labeled/sev/accuracy_stats.json
   Total: 2880
   Accepted: 2636
   Rejected: 244
   Overall Accuracy: 91.53%

   By Emotion:
      anger       :  475/ 480 =  98.96%
      disgust     :  342/ 480 =  71.25%
      fear        :  450/ 480 =  93.75%
      happiness   :  477/ 480 =  99.38%
      sadness     :  464/ 480 =  96.67%
      surprise    :  428/ 480 =  89.17%

   By Valence:
      negative    :  871/ 960 =  90.73%
      neutral     :  879/ 960 =  91.56%
      positive    :  886/ 960 =  92.29%

✅ Statistics generation completed!
